> **Course source:** [llama-index/components.ipynb](https://huggingface.co/agents-course/notebooks/blob/cecf8edffb02101c90634b84076bd93b1a20c713/unit2/llama-index/components.ipynb).
> **Compatibility:** Uses the original persona corpus, BGE embeddings and Chroma. The second query records local LlamaIndex trace events. The original Phoenix Cloud setup is retained as a reference and needs a separate account and key.

# Components in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

Alfred is hosting a party and needs to be able to find relevant information on personas that will be attending the party. Therefore, we will use a `QueryEngine` to index and search through a database of personas.

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
COURSE_PACKAGES = ['llama-index-core==0.14.26', 'llama-index-llms-openai-like==0.7.2', 'llama-index-embeddings-huggingface==0.9.0', 'llama-index-vector-stores-chroma==0.7.0', 'datasets==5.2.0']

import os
import subprocess
import sys

if not os.environ.get("HF_COURSE_BATCH"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", *COURSE_PACKAGES], check=True, timeout=600)

And, let's log in to Hugging Face to use serverless Inference APIs.

In [2]:
import json
import os
from functools import lru_cache

import httpx
from openai import OpenAI

GROQ_BASE = "https://api.groq.com/openai/v1"
QWEN = "qwen/qwen3.8-27b"
GPT_OSS = ("openai/gpt-oss-20b", "openai/gpt-oss-120b")


def optional_secret(name):
    """Read a notebook-authorized secret without printing it."""
    if os.getenv(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return None
    except Exception as exc:
        if type(exc).__name__ in {"SecretNotFoundError", "NotebookAccessError"}:
            return None
        raise


@lru_cache(maxsize=1)
def groq_account():
    key = optional_secret("GROQ_API_KEY")
    if not key:
        raise RuntimeError("Enable notebook access to the GROQ_API_KEY secret in Colab.")
    with OpenAI(api_key=key, base_url=GROQ_BASE, timeout=20, max_retries=0) as client:
        return key, frozenset(model.id for model in client.models.list().data)


class ModelRouting:
    def __init__(self, models, mode):
        self.models = list(models)
        self.mode = mode

    def requests(self, request):
        if request.method != "POST" or not request.url.path.endswith("/chat/completions"):
            yield None, request
            return
        body = json.loads(request.content)
        headers = {key: val for key, val in request.headers.items()
                   if key not in {"content-length", "transfer-encoding"}}
        if self.mode == "code" and body.get("tools"):
            raise ValueError("CodeAgent uses Python actions. Its API request must have no native tools.")
        for model_id in tuple(self.models):
            payload = {**body, "model": model_id}
            max_tokens = payload.pop("max_tokens", None)
            if max_tokens and "max_completion_tokens" not in payload:
                payload["max_completion_tokens"] = max_tokens
            if model_id in GPT_OSS:
                payload.pop("parallel_tool_calls", None)
                payload.pop("stop", None)
                payload["reasoning_effort"] = "low"
            elif model_id == QWEN:
                payload["reasoning_effort"] = "none"
            yield model_id, httpx.Request(
                request.method, request.url, json=payload, headers=headers,
                extensions=request.extensions,
            )

    def accept(self, model_id, status, attempt):
        if model_id is None:
            return True
        if status in {429, 503} and attempt < len(self.models):
            return False
        if 200 <= status < 300 and model_id != self.models[0]:
            self.models.remove(model_id)
            self.models.insert(0, model_id)
            print(f"Groq fallback model: {model_id}", flush=True)
        return True


class CourseTransport(httpx.HTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    def handle_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = super().handle_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            response.close()


class AsyncCourseTransport(httpx.AsyncHTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    async def handle_async_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = await super().handle_async_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            await response.aclose()


@lru_cache(maxsize=3)
def groq_route(mode):
    key, available = groq_account()
    candidates = {
        "text": (QWEN, *GPT_OSS),
        "vision": (QWEN,),
        "code": (QWEN, *GPT_OSS),
    }[mode]
    models = [model_id for model_id in candidates if model_id in available]
    if not models:
        raise RuntimeError(f"None of the Groq {mode} models are enabled for this project: {candidates}")
    print(f"Groq {mode} candidates, in order: {', '.join(models)}", flush=True)
    return key, ModelRouting(models, mode)


@lru_cache(maxsize=1)
def make_groq_llm():
    from llama_index.llms.openai_like import OpenAILike
    key, routing = groq_route("text")
    return OpenAILike(
        model=routing.models[0], api_base=GROQ_BASE, api_key=key,
        is_chat_model=True, is_function_calling_model=True, context_window=32768,
        max_tokens=1024, temperature=0.1, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        async_http_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=2)
def make_chat_model(mode="text"):
    from langchain_openai import ChatOpenAI
    key, routing = groq_route(mode)
    return ChatOpenAI(
        model=routing.models[0], base_url=GROQ_BASE, api_key=key,
        temperature=0.1, max_tokens=1024, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        http_async_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=1)
def make_code_model():
    from smolagents import OpenAIModel
    key, routing = groq_route("code")
    return OpenAIModel(
        model_id=routing.models[0], api_base=GROQ_BASE, api_key=key,
        max_tokens=1800, temperature=0.1, retry=False,
        client_kwargs={
            "http_client": httpx.Client(transport=CourseTransport(routing), timeout=30),
            "timeout": 30, "max_retries": 0,
        },
    )
from pathlib import Path
from hashlib import sha256

from datasets import load_dataset
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

PERSONAS = "dvilasuero/finepersonas-v0.1-tiny"


@lru_cache(maxsize=1)
def persona_embedding():
    return HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")


def persona_documents(folder, only_first_ten=False):
    dataset = load_dataset(PERSONAS, split="train")
    folder.mkdir(parents=True, exist_ok=True)
    indices = sorted(range(len(dataset)), key=lambda i: f"persona_{i}.txt")
    if only_first_ten:
        indices = indices[:10]
    files = []
    for i in indices:
        path = folder / f"persona_{i}.txt"
        path.write_text(dataset[i]["persona"], encoding="utf-8")
        files.append(str(path))
    return SimpleDirectoryReader(input_files=files, encoding="utf-8").load_data()


async def ingest_personas(collection, vector_store, documents):
    nodes = await IngestionPipeline(
        transformations=[SentenceSplitter()], 
    ).arun(documents=documents[:10])
    for node in nodes:
        node.id_ = sha256((node.metadata["file_name"] + "\n" + node.text).encode()).hexdigest()
    existing = set(collection.get(include=[])["ids"])
    missing = [node for node in nodes if node.node_id not in existing]
    if missing:
        new_nodes = await IngestionPipeline(
            transformations=[persona_embedding()], vector_store=vector_store,
        ).arun(nodes=missing)
        embeddings = {node.node_id: node.embedding for node in new_nodes}
    else:
        embeddings = {}
    if len(embeddings) != len(missing) or not collection.count():
        raise RuntimeError("The Chroma ingestion did not persist all expected persona nodes.")
    return nodes

## Create a `QueryEngine` for retrieval augmented generation

### Setting up the persona database

We will be using personas from the [dvilasuero/finepersonas-v0.1-tiny dataset](https://huggingface.co/datasets/dvilasuero/finepersonas-v0.1-tiny). This dataset contains 5K personas that will be attending the party!

Let's load the dataset and store it as files in the `data` directory

In [3]:
documents = persona_documents(Path("data"))
print("Course persona documents:", len(documents))

Course persona documents: 5000


Generating train split: 100%|##########| 5000/5000 [00:00<00:00, 15061.33 examples/s]


Awesome, now we have a local directory with all the personas that will be attending the party, we can load and index!

### Loading and embedding persona documents

We will use the `SimpleDirectoryReader` to load the persona descriptions from the `data` directory. This will return a list of `Document` objects. 

In [4]:
len(documents)

5000

Now we have a list of `Document` objects, we can use the `IngestionPipeline` to create nodes from the documents and prepare them for the `QueryEngine`. We will use the `SentenceSplitter` to split the documents into smaller chunks and the `HuggingFaceEmbedding` to embed the chunks.

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        persona_embedding(),
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes

Loading weights: 100%|##########| 199/199 [00:00<00:00, 4064.58it/s]


[TextNode(id_='66084828-2055-4854-8a49-f065500eabd2', embedding=[0.0064322915859520435, 0.0246534813195467, 0.04237119480967522, 0.0039515988901257515, -0.008177840150892735, -0.025908716022968292, 0.013009094633162022, 0.022640345618128777, -0.07412802428007126, -0.0504799447953701, 0.00023867344134487212, -0.03357742726802826, -0.055890634655952454, 0.03938386216759682, -0.03054220601916313, -0.002258863765746355, 0.0036918274126946926, 0.08876965939998627, 0.003260119818150997, 0.01283332984894514, 0.00926273874938488, -0.060987696051597595, 0.0357038676738739, -0.02436664327979088, -0.027047930285334587, 0.005153457168489695, 0.016612501814961433, -0.0076207686215639114, -0.020157719030976295, -0.1149883046746254, -0.0594494491815567, 0.00731232576072216, -0.023536691442131996, 0.0180167555809021, 0.0634373128414154, 0.009770307689905167, 0.017255211248993874, 0.07172060757875443, 0.0053900969214737415, 0.008163013495504856, 0.029093727469444275, 0.016303692013025284, -0.0338417626

As, you can see, we have created a list of `Node` objects, which are just chunks of text from the original documents. Let's explore how we can add these nodes to a vector store.

### Storing and indexing documents

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it.
In this case, we will use `Chroma` to store our documents.
Let's run the pipeline again with the vector store attached. 
The `IngestionPipeline` caches the operations so this should be fast!

In [6]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

nodes = await ingest_personas(chroma_collection, vector_store, documents)
len(nodes)

10

We can create a `VectorStoreIndex` from the vector store and use it to query the documents by passing the vector store and embedding model to the `from_vector_store()` method.

In [7]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


embed_model = persona_embedding()
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

We don't need to worry about persisting the index to disk, as it is automatically saved within the `ChromaVectorStore` object and the passed directory path.

### Querying the index

Now that we have our index, we can use it to query the documents.
Let's create a `QueryEngine` from the index and use it to query the documents using a specific response mode.


In [8]:
import nest_asyncio

nest_asyncio.apply()  # This is needed to run the query engine
llm = make_groq_llm()
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response = await query_engine.aquery(
    "Respond using a persona that describes author and travel experiences?"
)
response

Groq text candidates, in order: qwen/qwen3.8-27b, openai/gpt-oss-20b, openai/gpt-oss-120b


Response(response='I am an anthropologist or cultural expert with a deep interest in the intricacies of Cypriot culture, history, and society. I have spent considerable time researching and living in Cyprus, which has allowed me to gain a profound understanding of its people, customs, and way of life.', source_nodes=[NodeWithScore(node=TextNode(id_='60c903d7722aeaa000e353a84c3c7eb35cba2f1a41acc6efd77424842aef0480', embedding=None, metadata={'file_path': 'data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='68f7b1a0-5039-4171-8e05-1a5cac018fd5', node_type='4', metadata={'file_path': 'data/persona_1.txt', 'file_name': 

## Evaluation and observability

LlamaIndex provides **built-in evaluation tools to assess response quality.**
These evaluators leverage LLMs to analyze responses across different dimensions.
We can now check if the query is faithful to the original persona.

In [9]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = await evaluator.aevaluate_response(response=response)
eval_result.passing

Groq fallback model: openai/gpt-oss-20b


True

If one of these LLM based evaluators does not give enough context, we can check the response using the Arize Phoenix tool, after creating an account at [LlamaTrace](https://llamatrace.com/login) and generating an API key.

**Original Phoenix Cloud configuration (reference)**

```python
import llama_index
import os

PHOENIX_API_KEY = "<PHOENIX_API_KEY>"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
llama_index.core.set_global_handler(
    "arize_phoenix", endpoint="https://llamatrace.com/v1/traces"
)

```

In [10]:
from llama_index.core import Settings
from llama_index.core.callbacks import CallbackManager, CBEventType, LlamaDebugHandler

trace_handler = LlamaDebugHandler(print_trace_on_end=False)
Settings.callback_manager = CallbackManager([trace_handler])

Now, we can query the index and see the response in the Arize Phoenix tool.

In [11]:
query_engine = index.as_query_engine(llm=make_groq_llm(), response_mode="tree_summarize")
make_groq_llm().callback_manager = Settings.callback_manager
response = await query_engine.aquery(
    "What is the name of the someone that is interested in AI and techhnology?"
)
response
print("Completed local LLM callback events:", len(trace_handler.get_event_pairs(CBEventType.LLM)))
response

Completed local LLM callback events: 1


Response(response='There is no name provided for anyone interested in AI and technology.', source_nodes=[NodeWithScore(node=TextNode(id_='60c903d7722aeaa000e353a84c3c7eb35cba2f1a41acc6efd77424842aef0480', embedding=None, metadata={'file_path': 'data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='68f7b1a0-5039-4171-8e05-1a5cac018fd5', node_type='4', metadata={'file_path': 'data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266}, hash='65159e83029e97da100a6c7b0b78e10e123e11dca4c1b612d5c701ccb2d2d0d5')}, metadata_template='{key}: {value}', metadata_separator='\n', text='An anthro

We can then go to the [LlamaTrace](https://llamatrace.com/login) and explore the process and response.

![arize-phoenix](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/arize.png)    